# 05 · Eval harness from scratch

The "how do you know it works?" round, built from nothing. It has six pieces: a JSONL golden set with categories, a system under test, lexical metrics (exact match, token F1, contains-the-answer), an LLM judge with structured output that you **calibrate before you trust it**, bootstrap confidence intervals with a per-category breakdown, and a **paired regression test** between two prompt versions that decides ship or block. Results go to CSV.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [LLM eval metrics](../12_evaluation/01_llm_eval_metrics_em_f1_bleu_rouge.ipynb) and [LLM-as-judge](../12_evaluation/02_llm_as_judge_and_its_biases.ipynb) go deeper.

## Why this matters in interviews

- "How would you evaluate this?" follows every GenAI design answer. Writing a harness live shows you know evals are **code**: data, metrics, aggregation and a gate.
- The senior signals are **confidence intervals** instead of bare averages, a **per-category** breakdown instead of one number, a **paired** comparison between versions, and a judge that has been **calibrated against human labels**.
- The same harness becomes the CI gate that blocks a bad prompt change (`ev22`, `ev28`).

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ev24` | What is an eval harness, practically? |
| `ev05` | How do you build an eval set from scratch? |
| `ev15` | What is a golden dataset and how do you keep it useful? |
| `ev04` | Compare the main evaluation methods: exact match, semantic similarity, LLM judge, pairwise, RAGAS, human, and online A/B. |
| `ev03` | How do you make an LLM judge reliable? Name the biases and the fixes. |
| `ev19` | What is Cohen's kappa and why does it matter here? |
| `ev22` | What is regression testing for an LLM system, and how is it different? |
| `ev28` | How often should evals run, and what gates what? |
| `pr09` | How do you version and test prompts? |
| `ev31` | What is the minimum viable eval for a team shipping next week? |

## The question, as the interviewer asks it

> *"We have a support bot that answers from our help-centre docs. A teammate wants to ship a new prompt: it adds an 'I don't know' instruction and caps answers at 12 tokens to save money. Build me something that tells us whether to ship it."*

## The first 60 seconds

| Clarify | Why |
|---|---|
| What does "good" mean: correct, grounded, cited, concise, safe? | Each is a different metric. Pick 2–3 and write them down before looking at outputs. |
| Is there labelled data? | If not, build a golden set: 50–200 real questions with reference answers, stratified by category, including questions the bot *should refuse*. |
| Can answers be checked by string matching? | Short factual answers can. Open-ended ones need a judge or humans. |
| What change counts as a regression? | Decide the gate *before* running, for example "no category drops by more than 10 points". |

**The plan:** "Store the eval set as JSONL. Each row gets an id, a question, a reference and a category. Run each prompt version over it, score every output with cheap lexical metrics plus an LLM judge, but only after checking the judge against human labels. Report means with bootstrap CIs, overall and per category. Compare the versions *paired*, item by item, and turn that into a ship/block decision. Write everything to CSV so a run can be diffed later."

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

## Packages

| Package | What it gives you | Swap it for |
|---|---|---|
| `json` (stdlib) | the JSONL eval set: one JSON object per line, easy to append, diff and stream | a dataset table (Postgres), Hugging Face `datasets` |
| `openai` | the system under test and the judge (`chat.completions.parse` with a pydantic schema) | any model; a judge from a *different* family reduces self-preference |
| `pydantic` | the judge's output schema, so the verdict is always parseable | JSON mode plus manual validation |
| `numpy`, `scipy.stats` | bootstrap CIs, the sign test on paired outcomes | `statsmodels` |
| `sklearn.metrics` | `cohen_kappa_score` for judge-versus-human agreement | hand-rolled kappa (4 lines) |
| `pandas` | per-item tables, group-bys, CSV output | plain `csv` |
| *(frameworks)* | — | OpenAI Evals, promptfoo, DeepEval, RAGAS, LangSmith, Braintrust: the same ideas with dashboards |

## Step 1 · The golden set, as JSONL

Fifteen items in three categories. **`numbers`** ask for a figure. **`policy`** ask what is allowed and how. **`unanswerable`** are *not* in the docs, and the right answer is to decline. Every golden set needs that last category, or you never measure hallucination.

In [ ]:
import json
import re
import tempfile

import numpy as np

EVAL_ITEMS = [
    ("How many days do I have to ask for a refund?", "30 days", "numbers"),
    ("How long until a refund reaches my card?", "5 business days", "numbers"),
    ("How much does express shipping cost?", "$15", "numbers"),
    ("How long is the electronics warranty?", "2-year", "numbers"),
    ("When do password reset links expire?", "after 30 minutes", "numbers"),
    ("How long is an account locked after failed logins?", "15 minutes", "numbers"),
    ("How much is the Pro plan?", "$29 per month", "numbers"),
    ("How much do I save by paying yearly?", "20 percent", "numbers"),
    ("Can final sale items be refunded?", "cannot be refunded", "policy"),
    ("Which orders ship free?", "orders over $50", "policy"),
    ("Is accidental damage covered?", "not covered", "policy"),
    ("How do I make a warranty claim?", "through the support portal", "policy"),
    ("Who is the CEO?", "I don't know", "unanswerable"),
    ("Do you have a store in Paris?", "I don't know", "unanswerable"),
    ("What is the phone number for support?", "I don't know", "unanswerable"),
]
WORK = Path(tempfile.mkdtemp(prefix="evals_"))
EVAL_PATH = WORK / "golden_v1.jsonl"
with EVAL_PATH.open("w", encoding="utf-8") as f:
    for i, (q, ref, cat) in enumerate(EVAL_ITEMS):
        f.write(json.dumps({"id": f"q{i:02d}", "question": q, "reference": ref, "category": cat}) + "\n")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

GOLDEN = load_jsonl(EVAL_PATH)
print(EVAL_PATH.name, "->", len(GOLDEN), "items;", {c: sum(r["category"] == c for r in GOLDEN) for c in ["numbers", "policy", "unanswerable"]})
print(EVAL_PATH.read_text().splitlines()[0])

## Step 2 · The system under test: a tiny RAG, in two prompt configs

The system has five help-centre docs, BM25 retrieval of the top 2, and one chat call. A **prompt version is the template plus its generation parameters**. `v2` adds "say you don't know" (good) and caps output at 12 tokens (a cost saving with a risk). Keeping both versions as data (dicts) rather than code is what makes them comparable and loggable (`pr09`).

In [ ]:
from rank_bm25 import BM25Okapi

KB = {
    "refunds.md": "You can ask for a refund within 30 days of delivery. Refunds are paid back to the original card "
                  "within 5 business days. Items marked final sale cannot be refunded.",
    "shipping.md": "Standard shipping takes 3 to 5 business days. Express shipping costs $15 and arrives the next "
                   "business day. Orders over $50 ship free.",
    "warranty.md": "Electronics come with a 2-year warranty for manufacturing defects. Accidental damage is not covered "
                   "by the warranty. To make a warranty claim, upload the serial number and a photo through the support portal.",
    "account.md": "Password reset links expire after 30 minutes. After 5 failed login attempts the account is locked for 15 minutes.",
    "plans.md": "The Basic plan costs $9 per month. The Pro plan costs $29 per month and includes priority support. "
                "Paying yearly saves 20 percent.",
}
DOC_NAMES = list(KB)
tokenize = lambda t: re.findall(r"[a-z0-9$]+", t.lower())
bm25 = BM25Okapi([tokenize(KB[n]) for n in DOC_NAMES])

def retrieve(question, k=2):
    scores = bm25.get_scores(tokenize(question))
    return [DOC_NAMES[i] for i in np.argsort(-scores, kind="stable")[:k]]

PROMPTS = {
    "v1": {"system": "Answer the question using the context. Cite the source file in brackets.", "max_tokens": None},
    "v2": {"system": "Answer using only the context. Cite the source file in brackets. "
                     "If the context does not contain the answer, say you don't know.", "max_tokens": 12},
}

def run_system(question, version):
    cfg = PROMPTS[version]
    sources = retrieve(question)
    context = "\n".join(f"[{name}] {KB[name]}" for name in sources)
    extra = {"max_tokens": cfg["max_tokens"]} if cfg["max_tokens"] else {}
    resp = client.chat.completions.create(model=MODEL, temperature=0, **extra, messages=[
        {"role": "system", "content": cfg["system"]},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    return {"output": resp.choices[0].message.content, "sources": sources,
            "finish_reason": resp.choices[0].finish_reason, "tokens": resp.usage.total_tokens,
            "prompt_tokens": resp.usage.prompt_tokens, "completion_tokens": resp.usage.completion_tokens}

OUTPUTS = {v: {row["id"]: run_system(row["question"], v) for row in GOLDEN} for v in PROMPTS}
for row in GOLDEN[:2] + GOLDEN[11:13]:
    print(f"{row['id']} {row['question']}")
    for v in PROMPTS:
        o = OUTPUTS[v][row["id"]]
        print(f"   {v} [{o['finish_reason']:6}] {o['output'][:105]}")

Offline, the stand-in answers extractively. It copies the best-matching context sentence, which makes the lexical metrics meaningful. It also honours `max_tokens` (`finish_reason="length"`) and the "say you don't know" rule, which is why the two versions really do differ. Look at `q11`: v2's 12-token cap cuts the answer off before "through the support portal". A real model phrases its answers differently, but it reacts to both changes in the same direction.

## Step 3 · Metrics

| Metric | What it checks | Good for | Blind spot |
|---|---|---|---|
| **exact match** | normalised prediction == normalised reference | short-form answers (a label, a number) | any extra word fails it |
| **token F1** | word overlap between prediction and reference (SQuAD-style) | partial credit | rewards wordy answers that contain the words, punishes paraphrases |
| **correct** (`contains` / abstain) | the reference string appears in the output, or, for unanswerable items, the output declines | short facts inside sentences | misses paraphrases ("fifteen dollars") |
| **cited** | a `[file.md]` tag from the retrieved sources is present | groundedness hygiene | a citation is not proof of support |
| **LLM judge** | a model grades correctness against the reference | paraphrases, open-ended answers | bias, cost, drift: calibrate it first (Step 4) |

In [ ]:
from collections import Counter

ABSTAIN = re.compile(r"don.?t know|do not know|not (?:in|covered by) the (?:provided )?context|no information|"
                     r"cannot (?:find|answer)|does not (?:say|mention|contain)", re.I)

def normalize(text):
    text = re.sub(r"\[[^\]]*\]", " ", text.lower())             # drop citation tags
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(re.findall(r"[a-z0-9$%]+", text))

def exact_match(pred, ref):
    return float(normalize(pred) == normalize(ref))

def token_f1(pred, ref):
    p, r = normalize(pred).split(), normalize(ref).split()
    common = sum((Counter(p) & Counter(r)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(r)
    return 2 * precision * recall / (precision + recall)

def correct(pred, ref, category):
    if category == "unanswerable":
        return float(bool(ABSTAIN.search(pred)))
    return float(normalize(ref) in normalize(pred))

def cited(pred, sources):
    return float(any(f"[{s}]" in pred for s in sources))

assert exact_match("The answer is: $15.", "$15") == 0.0 and correct("It costs $15 [shipping.md]", "$15", "numbers") == 1.0
assert token_f1("30 days", "30 days") == 1.0 and 0 < token_f1("within 30 days of delivery", "30 days") < 1
assert correct("I don't know - not in the context.", "I don't know", "unanswerable") == 1.0

## Step 4 · An LLM judge, and calibrating it before you trust it

The judge returns a **pydantic object**: `reasoning` first (a brief justification helps), then a two-value `verdict`. With `chat.completions.parse` the output is guaranteed to fit the schema.

The judge is itself a model, so it can be wrong or biased: position, verbosity and self-preference (`ev03`). **Before using it, measure it against human labels** on a small fixed calibration set. Include clear passes, clear fails and paraphrases. Report **Cohen's kappa**, which is agreement corrected for chance (`ev19`), and gate on it. A judge that says "correct" to everything gets high raw agreement on a mostly-correct set and a kappa of zero.

In [ ]:
from typing import Literal

from pydantic import BaseModel
from sklearn.metrics import cohen_kappa_score

class Verdict(BaseModel):
    reasoning: str
    verdict: Literal["correct", "incorrect"]

JUDGE_SYSTEM = ("You grade a support bot. Given a question, a reference answer and a candidate answer, decide whether "
                "the candidate states the same facts as the reference. Paraphrases are fine; wrong or missing facts are not. "
                "If the reference says the bot should not know, the candidate must decline to answer.")

def judge(question, reference, candidate):
    resp = client.chat.completions.parse(model=MODEL, temperature=0, response_format=Verdict, messages=[
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": f"Question: {question}\nReference answer: {reference}\nCandidate answer: {candidate}"}])
    return resp.choices[0].message.parsed

CALIBRATION = [  # (question, reference, candidate, human label: 1 = correct)
    ("How much is the Pro plan?", "$29 per month", "The Pro plan is $29 a month.", 1),
    ("How much is the Pro plan?", "$29 per month", "It costs $9 per month.", 0),
    ("How many days do I have to ask for a refund?", "30 days", "Within 30 days of delivery.", 1),
    ("How many days do I have to ask for a refund?", "30 days", "You have 14 days.", 0),
    ("Is accidental damage covered?", "not covered", "No, accidental damage isn't covered.", 1),
    ("Is accidental damage covered?", "not covered", "Yes, the warranty covers it.", 0),
    ("Who is the CEO?", "I don't know", "I don't know - the documents don't say.", 1),
    ("Who is the CEO?", "I don't know", "The CEO is Jane Smith.", 0),
    ("How much does express shipping cost?", "$15", "Express delivery is fifteen dollars.", 1),
    ("How much does express shipping cost?", "$15", "Express shipping is free.", 0),
]
human = [label for *_, label in CALIBRATION]
judged = [int(judge(q, ref, cand).verdict == "correct") for q, ref, cand, _ in CALIBRATION]
agreement = np.mean(np.array(human) == np.array(judged))
kappa = cohen_kappa_score(human, judged)
JUDGE_TRUSTED = bool(kappa >= 0.6)
print(f"judge verdicts: {judged}\nhuman labels  : {human}")
print(f"raw agreement {agreement:.0%}, Cohen's kappa {kappa:.2f} -> judge trusted: {JUDGE_TRUSTED}")

**Offline, this gate does its job.** The stand-in fills the schema with the first allowed value, so it says `"correct"` to everything. It agrees with the humans on exactly the five correct items, which is 50% raw agreement and **kappa 0**. The harness therefore *refuses to use it*, and the lexical metrics carry the decision below. With `OPENAI_API_KEY` set, `gpt-4.1-mini` typically agrees with these clear-cut labels on most or all items (kappa well above 0.6), and the judge column switches on automatically. The code is the same either way. Only the evidence changes.

Two more judge practices: re-run the calibration whenever the judge model or prompt changes (judge drift, `ev34`), and for A/B comparisons prefer **pairwise** judging with both orders (A,B) and (B,A) to cancel position bias.

## Step 5 · Score everything, per item

In [ ]:
import pandas as pd

rows = []
for version in PROMPTS:
    for item in GOLDEN:
        out = OUTPUTS[version][item["id"]]
        pred = out["output"]
        rows.append({
            "version": version, "id": item["id"], "category": item["category"], "question": item["question"],
            "reference": item["reference"], "output": pred, "finish_reason": out["finish_reason"], "tokens": out["tokens"],
            "prompt_tokens": out["prompt_tokens"], "completion_tokens": out["completion_tokens"],
            "exact_match": exact_match(pred, item["reference"]),
            "token_f1": token_f1(pred, item["reference"]),
            "correct": correct(pred, item["reference"], item["category"]),
            "cited": cited(pred, out["sources"]) if item["category"] != "unanswerable" else np.nan,
            "judge": (float(judge(item["question"], item["reference"], pred).verdict == "correct")
                      if JUDGE_TRUSTED else np.nan),
        })
RESULTS = pd.DataFrame(rows)
print(RESULTS.groupby("version")[["exact_match", "token_f1", "correct", "cited", "judge", "tokens"]].mean().round(3))

Exact match is 0 across the board, because full-sentence answers never equal a 2-word reference. That is the usual reason EM is the wrong primary metric for chat-style output. `correct` and `token_f1` carry the signal here. Note that `cited` dropped for v2: the 12-token cap cuts off the `[file.md]` tag at the end of long answers.

## Step 6 · Aggregate with bootstrap confidence intervals

With 15 items, each item moves the mean by almost 7 points. A bare "73% vs 80%" is meaningless without its uncertainty. The **percentile bootstrap** resamples items with replacement, recomputes the mean each time, and reads the 2.5th and 97.5th percentiles.

In [ ]:
def bootstrap_ci(values, n_boot=4000, seed=0):
    values = np.asarray(values, float)
    values = values[~np.isnan(values)]
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return values.mean(), *np.percentile(means, [2.5, 97.5])

summary = []
for version in PROMPTS:
    for scope in ["ALL", "numbers", "policy", "unanswerable"]:
        sub = RESULTS[(RESULTS.version == version) & ((RESULTS.category == scope) | (scope == "ALL"))]
        mean, lo, hi = bootstrap_ci(sub["correct"])
        summary.append({"version": version, "scope": scope, "n": len(sub), "correct": mean, "ci_low": lo, "ci_high": hi})
SUMMARY = pd.DataFrame(summary)
print(SUMMARY.round(2).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

scopes = ["ALL", "numbers", "policy", "unanswerable"]
x = np.arange(len(scopes))
fig, ax = plt.subplots(figsize=(9, 3.8))
for j, (version, color) in enumerate([("v1", "#adb5bd"), ("v2", "#4c6ef5")]):
    s = SUMMARY[SUMMARY.version == version].set_index("scope").loc[scopes]
    ax.bar(x + (j - 0.5) * 0.36, s["correct"], 0.36, color=color, label=version,
           yerr=[s["correct"] - s["ci_low"], s["ci_high"] - s["correct"]], capsize=4)
ax.set_xticks(x, [f"{sc}\n(n={int(SUMMARY[(SUMMARY.version == 'v1') & (SUMMARY.scope == sc)].n.iloc[0])})" for sc in scopes])
ax.set_ylim(0, 1.15); ax.set_ylabel("correct (95% bootstrap CI)")
ax.set_title("Accuracy by category: the overall number hides a win and a loss"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

The per-category view is the point of this chart. **Unanswerable** improves: v2 declines where v1 hallucinated or rambled. **Policy** gets worse: the length cap truncated an answer. The error bars on 3–8 items per category are wide, so one run is not proof of anything. That is exactly why the next step compares the versions *paired*.

## Step 7 · The paired regression test: ship or block?

Both versions answered **the same questions**, so compare them item by item. Paired comparisons remove the between-question variance, which is most of the noise, so they detect smaller changes with the same data.

- **Wins / losses / ties** per item.
- A **paired bootstrap** CI on the mean difference (resample *item ids*, keeping each item's v1 and v2 together).
- An exact **sign test** (`scipy.stats.binomtest`) on the discordant items. If the versions were equally good, wins and losses would be a fair coin.
- **The gate:** block if any category's accuracy drops by more than 10 points, whatever the overall number says. Decide this rule *before* looking at the results.

In [ ]:
from scipy.stats import binomtest

wide = RESULTS.pivot(index="id", columns="version", values="correct")
wide["category"] = RESULTS[RESULTS.version == "v1"].set_index("id")["category"]
diff = (wide["v2"] - wide["v1"]).to_numpy()
wins, losses = int((diff > 0).sum()), int((diff < 0).sum())
mean_diff, lo, hi = bootstrap_ci(diff)
p = binomtest(wins, wins + losses, 0.5).pvalue if wins + losses else 1.0
print(f"paired: v2 wins {wins}, loses {losses}, ties {len(diff) - wins - losses}")
print(f"mean difference {mean_diff:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  sign-test p = {p:.2f}")

per_cat = (wide.groupby("category")[["v1", "v2"]].mean().assign(delta=lambda d: d.v2 - d.v1))
print("\n", per_cat.round(2))
print("\nlosses:", list(wide.index[diff < 0]), "| wins:", list(wide.index[diff > 0]))

GATE_MAX_CATEGORY_DROP = 0.10
blocked = per_cat[per_cat.delta < -GATE_MAX_CATEGORY_DROP]
decision = "BLOCK" if len(blocked) else "SHIP"
print(f"\nGATE (no category may drop > {GATE_MAX_CATEGORY_DROP:.0%}): {decision}", list(blocked.index))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
colors = ["#2f9e44" if d > 0 else "#e03131" if d < 0 else "#dee2e6" for d in diff]
ax.bar(range(len(diff)), np.where(diff == 0, 0.05, diff), color=colors)
ax.set_xticks(range(len(diff)), [f"{i}\n{c[:4]}" for i, c in zip(wide.index, wide.category)], fontsize=7)
ax.axhline(0, color="black", lw=0.8); ax.set_ylim(-1.2, 1.2); ax.set_ylabel("v2 - v1 (correct)")
ax.set_title(f"Per-item paired outcome: {wins} wins (green), {losses} losses (red), grey = tie  ->  {decision}")
plt.tight_layout(); plt.show()

How to read the verdict:

- The **overall** difference may look neutral or even positive, and its CI straddles zero at n = 15. Averaged alone, it would wave the change through.
- The **per-category gate** catches the policy regression: the 12-token cap truncated an answer. The fix is obvious once you look at the losing item, for example cap at 60 tokens, or check `finish_reason == "length"` as its own metric.
- With a real golden set of 100–300 items the CIs tighten, and the sign test gains the power to call smaller changes.

## Step 8 · Write the run to CSV

Store every run with its version, timestamp and golden-set name, so a result from three weeks ago can be reproduced and diffed (`po31`).

In [ ]:
from datetime import datetime, timezone

run_id = f"{EVAL_PATH.stem}__{'_vs_'.join(PROMPTS)}__{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
items_csv, summary_csv = WORK / f"{run_id}__items.csv", WORK / f"{run_id}__summary.csv"
RESULTS.assign(model=MODEL, run_id=run_id).to_csv(items_csv, index=False)
SUMMARY.assign(model=MODEL, run_id=run_id, decision=decision).to_csv(summary_csv, index=False)
back = pd.read_csv(items_csv)
print(items_csv.name, back.shape, "|", summary_csv.name, pd.read_csv(summary_csv).shape)
print(back[["version", "id", "category", "correct", "token_f1", "finish_reason"]].head(4).to_string(index=False))
assert len(back) == len(GOLDEN) * len(PROMPTS)

## The whole harness as one file: `evals.py`

```python
"""evals.py - golden set in, per-category scores with CIs and a ship/block decision out.
pip install openai pydantic numpy pandas scipy scikit-learn ; env: OPENAI_API_KEY
usage: python evals.py golden.jsonl"""
import json
import re
import sys
from collections import Counter
from typing import Literal

import numpy as np
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel
from scipy.stats import binomtest

client, MODEL = OpenAI(), "gpt-4.1-mini"
PROMPTS = {
    "v1": {"system": "Answer the question using the context. Cite the source file in brackets.", "max_tokens": None},
    "v2": {"system": "Answer using only the context. Cite the source file in brackets. "
                     "If the context does not contain the answer, say you don't know.", "max_tokens": 12},
}
ABSTAIN = re.compile(r"don.?t know|do not know|not (?:in|covered by) the (?:provided )?context|no information", re.I)

def run_system(question, version, context):          # replace with a call to your real system
    cfg = PROMPTS[version]
    extra = {"max_tokens": cfg["max_tokens"]} if cfg["max_tokens"] else {}
    resp = client.chat.completions.create(model=MODEL, temperature=0, **extra, messages=[
        {"role": "system", "content": cfg["system"]},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    return resp.choices[0].message.content

def normalize(text):
    text = re.sub(r"\[[^\]]*\]", " ", text.lower())
    return " ".join(re.findall(r"[a-z0-9$%]+", re.sub(r"\b(a|an|the)\b", " ", text)))

def token_f1(pred, ref):
    p, r = normalize(pred).split(), normalize(ref).split()
    common = sum((Counter(p) & Counter(r)).values())
    return 0.0 if common == 0 else 2 * common / (len(p) + len(r))

def correct(pred, ref, category):
    return float(bool(ABSTAIN.search(pred))) if category == "unanswerable" else float(normalize(ref) in normalize(pred))

class Verdict(BaseModel):
    reasoning: str
    verdict: Literal["correct", "incorrect"]

def judge(question, reference, candidate):
    resp = client.chat.completions.parse(model=MODEL, temperature=0, response_format=Verdict, messages=[
        {"role": "system", "content": "Decide whether the candidate states the same facts as the reference."},
        {"role": "user", "content": f"Question: {question}\nReference answer: {reference}\nCandidate answer: {candidate}"}])
    return resp.choices[0].message.parsed.verdict == "correct"

def bootstrap_ci(values, n_boot=4000, seed=0):
    values = np.asarray(values, float)
    means = np.random.default_rng(seed).choice(values, size=(n_boot, len(values))).mean(axis=1)
    return values.mean(), *np.percentile(means, [2.5, 97.5])

if __name__ == "__main__":
    golden = [json.loads(line) for line in open(sys.argv[1], encoding="utf-8") if line.strip()]
    rows = []
    for version in PROMPTS:
        for item in golden:                            # each item: id, question, reference, category, context
            pred = run_system(item["question"], version, item.get("context", ""))
            rows.append({"version": version, **item, "output": pred,
                         "token_f1": token_f1(pred, item["reference"]),
                         "correct": correct(pred, item["reference"], item["category"])})
    df = pd.DataFrame(rows)
    for version, sub in df.groupby("version"):
        mean, lo, hi = bootstrap_ci(sub["correct"])
        print(f"{version}: correct {mean:.2f} [{lo:.2f}, {hi:.2f}]")
    wide = df.pivot(index="id", columns="version", values="correct")
    diff = (wide["v2"] - wide["v1"]).to_numpy()
    wins, losses = int((diff > 0).sum()), int((diff < 0).sum())
    print("paired diff CI:", bootstrap_ci(diff), "sign test p:", binomtest(wins, max(wins + losses, 1)).pvalue)
    per_cat = df.pivot_table(index="category", columns="version", values="correct")
    blocked = per_cat[(per_cat["v2"] - per_cat["v1"]) < -0.10]
    print(per_cat.round(2), "\nDECISION:", "BLOCK " + str(list(blocked.index)) if len(blocked) else "SHIP")
    df.to_csv("eval_results.csv", index=False)
```

## Try it yourself

**1.** Fix v2 by raising the cap to 60 tokens (call it `v3`), re-run the paired comparison **v1 vs v3**, and see what the gate says now.

In [ ]:
# Solution — try it yourself first, then run this
PROMPTS["v3"] = {**PROMPTS["v2"], "max_tokens": 60}
OUTPUTS["v3"] = {row["id"]: run_system(row["question"], "v3") for row in GOLDEN}
v1c = np.array([correct(OUTPUTS["v1"][r["id"]]["output"], r["reference"], r["category"]) for r in GOLDEN])
v3c = np.array([correct(OUTPUTS["v3"][r["id"]]["output"], r["reference"], r["category"]) for r in GOLDEN])
cats = np.array([r["category"] for r in GOLDEN])
deltas = {c: v3c[cats == c].mean() - v1c[cats == c].mean() for c in ["numbers", "policy", "unanswerable"]}
print("v3 - v1 by category:", {c: round(d, 2) for c, d in deltas.items()})
print("GATE:", "BLOCK" if min(deltas.values()) < -GATE_MAX_CATEGORY_DROP else "SHIP")
del PROMPTS["v3"]

**2.** Implement Cohen's kappa yourself (observed agreement versus the agreement expected by chance from each rater's label frequencies) and check it against `sklearn` on the calibration labels, plus on a judge that agrees 9 times out of 10.

In [ ]:
# Solution — try it yourself first, then run this
def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    p_obs = np.mean(a == b)
    p_exp = sum(np.mean(a == k) * np.mean(b == k) for k in np.union1d(a, b))
    return 0.0 if p_exp == 1 else (p_obs - p_exp) / (1 - p_exp)

good_judge = human.copy(); good_judge[3] = 1 - good_judge[3]          # disagrees once
for name, labels in [("stand-in / this run", judged), ("a 90%-agreeing judge", good_judge)]:
    print(f"{name:22} ours={kappa(human, labels):.3f}  sklearn={cohen_kappa_score(human, labels):.3f}")
    assert abs(kappa(human, labels) - cohen_kappa_score(human, labels)) < 1e-9

**3.** v2's 12-token cap was sold as a cost saving. Check that claim: total prompt tokens and completion tokens per version, and the change in total. A regression gate usually weighs quality *and* cost (`ev23`).

In [ ]:
# Solution — try it yourself first, then run this
cost = RESULTS.groupby("version")[["prompt_tokens", "completion_tokens", "tokens"]].sum()
print(cost.to_string())
print(f"\ntotal tokens, v2 vs v1: {cost.loc['v2', 'tokens'] / cost.loc['v1', 'tokens'] - 1:+.1%}")
# In a RAG call the *prompt* (instructions + retrieved context) dominates the bill. Capping the answer
# trims the small part, and v2's longer system prompt adds to the big part - so the "saving" may be none.
# Measure cost claims the same way you measure quality claims.

## Say it in an interview

**30 seconds (`ev24`):** "A harness is four things. First a versioned golden set: JSONL with categories, including questions the system should refuse. Then a runner that executes a prompt version, meaning template plus parameters, over it and logs every output. Then metrics: cheap lexical ones first, and an LLM judge only after I've checked its kappa against human labels. Finally aggregation: means with bootstrap CIs, per category, and a paired comparison against the current version, with a gate that was decided before the run, such as 'no category drops more than 10 points'. It runs in CI on every prompt change and writes CSVs, so any result is reproducible."

**Numbers:** 50–200 golden items to start. At n = 15, one item is about 7 points. A judge needs kappa ≥ 0.6 ("substantial") before it gates anything. Use about 4,000 bootstrap resamples. Paired tests beat unpaired ones on the same items.

**Traps:** one overall number hiding a category regression; EM on sentence answers; a judge nobody calibrated; the same model judging its own outputs (self-preference); changing the golden set and the prompt at the same time; no "should refuse" items; eyeballing five outputs and calling it an eval.

## Next

- Deeper: [EM, F1, BLEU, ROUGE](../12_evaluation/01_llm_eval_metrics_em_f1_bleu_rouge.ipynb) · [LLM-as-judge and its biases](../12_evaluation/02_llm_as_judge_and_its_biases.ipynb) · [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb) · [agent evaluation](../12_evaluation/04_agent_evaluation.ipynb) · [A/B testing and statistics for evals](../12_evaluation/05_ab_testing_and_statistics_for_evals.ipynb)
- Other rounds: [RAG skeleton](01_rag_skeleton.ipynb) · [ML model end to end](06_ml_model_end_to_end.ipynb)
- Theory: [ai_system_design_concepts course](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)